# Feature Selection

This notebook performs feature selection to identify key variables for econometric analysis.

**Methods:**
1. Correlation filtering (feature-outcome correlation)
2. VIF screening (multicollinearity detection)
3. Lasso regularization (sparse feature selection)
4. Stepwise selection (AIC/BIC criterion)
5. Ensemble compilation (union of selected features)

In [ ]:
# Import package and load processed data
from asean_green_bonds import data, config
import pandas as pd
import numpy as np

print('Loading processed data...')
df = data.load_processed_data(which='engineered')
print(f'Data shape: {df.shape}')
print(f'Columns: {df.shape[1]}')

In [ ]:
# Define outcome and control variables
outcomes = config.OUTCOME_VARIABLES
controls = config.CONTROL_VARIABLES
lagged = config.LAGGED_VARIABLES

print(f'Outcome variables: {len(outcomes)}')
print(outcomes)

print(f'\nCore control variables: {len(controls)}')
print(controls)

print(f'\nLagged variables: {len(lagged)}')
for v in lagged[:5]:
    print(f'  - {v}')
print(f'  ... and {len(lagged)-5} more')

In [ ]:
# Calculate VIF for initial features
print('Checking multicollinearity (VIF)...')

vif_df = data.calculate_vif(
    df,
    exclude_cols=['ric', 'Year', 'country', 'gic', 'green_bond_issue']
)

print('\nVariance Inflation Factors:')
print(vif_df.head(15))

high_vif = vif_df[vif_df['VIF'] > 10]
print(f'\n⚠️  Features with VIF > 10: {len(high_vif)}')
for _, row in high_vif.iterrows():
    print(f'  - {row["Variable"]}: {row["VIF"]:.2f}')

In [ ]:
# Run full feature selection pipeline
print('Running feature selection...')

selected_features, report = data.compile_selected_features(
    df,
    outcome_cols=outcomes,
    control_cols=controls,
    lagged_cols=lagged,
    selection_method='union'
)

print(f'\nSelected {len(selected_features)} features')
print('\nSelected Features:')
for feat in selected_features:
    print(f'  - {feat}')

In [ ]:
# Generate detailed report
print('Creating feature selection report...')

report_df = data.create_feature_selection_report(
    df,
    selected_features,
    outcomes
)

print('\nFeature Summary:')
print(report_df[['Feature', 'Non_Missing_Count', 'Missing_Pct', 'Mean', 'Std']].head(10))

print('\nCorrelations with outcomes:')
corr_cols = [c for c in report_df.columns if c.startswith('Corr_with_')]
print(report_df[['Feature'] + corr_cols].head(10))

In [ ]:
# Save selected dataset
print('Saving selected feature dataset...')

# Create dataset with selected features only
id_cols = ['ric', 'Year', 'country', 'gic']
treatment_cols = ['green_bond_issue', 'green_bond_active', 'is_certified', 'certified_bond_active']
selected_cols = id_cols + treatment_cols + selected_features

# Only keep available columns
available_cols = [c for c in selected_cols if c in df.columns]
df_selected = df[available_cols].copy()

# Save
output_path = config.PROCESSED_DATA_FILES['selected_features']
df_selected.to_csv(output_path, index=False)

print(f'✅ Saved to {output_path}')
print(f'   Shape: {df_selected.shape}')
print(f'   Columns: {df_selected.shape[1]}')